In [2]:
# import pandas as pd

# # 1. Baca data per 100.000 baris
# chunks = []
# for chunk in pd.read_csv('../data/raw/flights.csv', chunksize=100000, low_memory=False):
#     filtered = chunk[chunk['MONTH'].isin([1, 2])].copy()
#     chunks.append(filtered)

# # 2. Gabungan hasil filter bulan 1 & 2
# df_2m = pd.concat(chunks, ignore_index=True)

# # 3. membersihkan data kosong & buat target is_delay
# df_2m = df_2m.dropna(subset=['DEPARTURE_DELAY'])
# df_2m['is_delay'] = (df_2m['DEPARTURE_DELAY'] >= 15).astype(int)

# # 4. Simpan ke file subset baru
# df_2m.to_csv(('../data/raw/flights_2months.csv'), index=False)

# print("Selesai! Total baris data 2 bulan:", len(df_2m))
# print("Jumlah penerbangan delay:", df_2m['is_delay'].sum())



In [3]:
#Load dataser yang 2 bulan
import pandas as pd

df = pd.read_csv(('../data/raw/flights_2months.csv'), low_memory=False)

df.info

<bound method DataFrame.info of         YEAR  MONTH  DAY  DAY_OF_WEEK AIRLINE  FLIGHT_NUMBER TAIL_NUMBER  \
0       2015      1    1            4      AS             98      N407AS   
1       2015      1    1            4      AA           2336      N3KUAA   
2       2015      1    1            4      US            840      N171US   
3       2015      1    1            4      AA            258      N3HYAA   
4       2015      1    1            4      AS            135      N527AS   
...      ...    ...  ...          ...     ...            ...         ...   
867438  2015      2   28            6      B6           1503      N593JB   
867439  2015      2   28            6      US           1770      N801AW   
867440  2015      2   28            6      US            467      N601AW   
867441  2015      2   28            6      F9            300      N223FR   
867442  2015      2   28            6      F9            422      N926FR   

       ORIGIN_AIRPORT DESTINATION_AIRPORT  SCHEDULED_DE

In [4]:
# 1. kolom penting yang dipakai buat Machine Learning
selected_cols = [
    'MONTH', 'DAY_OF_WEEK', 'AIRLINE', 
    'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 
    'SCHEDULED_DEPARTURE', 'DISTANCE', 'is_delay'
]

df_clean = df[selected_cols].copy()

# 2. Cek data kosong 
print("--- Jumlah Data Kosong Per Kolom ---")
print(df_clean.isnull().sum())

# 3. Intip 5 baris pertama data pilihan
df_clean.head()


--- Jumlah Data Kosong Per Kolom ---
MONTH                  0
DAY_OF_WEEK            0
AIRLINE                0
ORIGIN_AIRPORT         0
DESTINATION_AIRPORT    0
SCHEDULED_DEPARTURE    0
DISTANCE               0
is_delay               0
dtype: int64


,MONTH,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DISTANCE,is_delay
0,1,4,AS,ANC,SEA,5,1448,0
1,1,4,AA,LAX,PBI,10,2330,0
2,1,4,US,SFO,CLT,20,2296,0
3,1,4,AA,LAX,MIA,20,2342,0
4,1,4,AS,SEA,ANC,25,1448,0


In [5]:
# 1. Ekstrak jam keberangkatan (jam 1430 jadi jam 14)
df_clean['dep_hour'] = df_clean['SCHEDULED_DEPARTURE'] // 100

# 2. Ubah kolom teks jadi tipe category
cat_cols = ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')

# 3. Pilih fitur akhir yang dipakai untuk model ML
features = ['MONTH', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'dep_hour', 'DISTANCE']

X = df_clean[features]
y = df_clean['is_delay']

print("Fitur siap! Ukuran dataset:", X.shape)
X.dtypes




Fitur siap! Ukuran dataset: (867443, 7)


MONTH                     int64
DAY_OF_WEEK               int64
AIRLINE                category
ORIGIN_AIRPORT         category
DESTINATION_AIRPORT    category
dep_hour                  int64
DISTANCE                  int64
dtype: object

In [9]:
import mlflow
import mlflow.lightgbm
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score

# 1. Tentukan X (fitur) dan y (target)
X = df_clean[features]
y = df_clean['is_delay']

# 2. Bagi data: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Pakai SQLite backend lokal (Solusi paling stabil di Windows)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Flight_Delay_Prediction")

with mlflow.start_run(run_name="LightGBM_Baseline_Model"):
    params = {"n_estimators": 100, "random_state": 42}
    mlflow.log_params(params)
    
    # Latih model LightGBM
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    
    # Prediksi dan evaluasi
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    roc_auc = round(roc_auc_score(y_test, y_prob), 4)
    
    # Log metrik akurasi ke MLflow
    mlflow.log_metric("roc_auc_score", roc_auc)
    
    print("=== Hasil Evaluasi Model ML & MLflow Logging Sukses ===")
    print("ROC-AUC Score:", roc_auc)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


2026/10/25 23:09:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/25 23:09:27 INFO mlflow.store.db.utils: Updating database tables
2026/10/25 23:10:03 INFO mlflow.tracking.fluent: Experiment with name 'Flight_Delay_Prediction' does not exist. Creating a new experiment.


[LightGBM] [Info] Number of positive: 145813, number of negative: 548141
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.227014 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 814
[LightGBM] [Info] Number of data points in the train set: 693954, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.210119 -> initscore=-1.324208
[LightGBM] [Info] Start training from score -1.324208
=== Hasil Evaluasi Model ML & MLflow Logging Sukses ===
ROC-AUC Score: 0.7036

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.99      0.88    137036
           1       0.57      0.05      0.08     36453

    accuracy                           0.79    173489
   macro avg       0.68      0.52      0.48    173489
weighted avg       0.75      0.79      0.72    173489



In [ ]:
import joblib

# Simpan model langsung ke folder models
joblib.dump(model, '../models/lgb_model.joblib')
print("Model berhasil disimpan di '../models/lgb_model.joblib'!")


Model berhasil disimpan di '../models/lgb_model.joblib'!
